# Qwen3-8B → BPMN : benchmark Kaggle

Exécuter avec **Internet ON**, un **GPU** et un dépôt GitHub à jour (`main` une fois la PR fusionnée, ou la branche corrigée ci-dessous). Le benchmark affiche chaque cas et sauvegarde le rapport progressivement. Aucun layout Node n'est nécessaire pour mesurer la fidélité sémantique.

In [ ]:
import os, sys, subprocess, pathlib, shutil, time, urllib.request, json
REPO = pathlib.Path("/kaggle/working/workflow-BPMN-generator")
BRANCH = "fix/kaggle-qwen-evaluation"
if not (REPO / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "-b", BRANCH,
                    "https://github.com/as7ia/workflow-BPMN-generator.git", str(REPO)], check=True)
else:
    print("Dépôt existant :", REPO, "- vérifier la branche et les correctifs")
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e", str(REPO)], check=True)
print("Projet prêt")

In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "zstd", "curl"], check=True)
OLLAMA = shutil.which("ollama")
if not OLLAMA:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, executable="/bin/bash", check=True)
    OLLAMA = shutil.which("ollama") or "/usr/local/bin/ollama"
if not pathlib.Path(OLLAMA).is_file():
    raise RuntimeError("Ollama introuvable après installation")
subprocess.run([OLLAMA, "--version"], check=True)
print("Ollama :", OLLAMA)

In [ ]:
def ollama_ready():
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if not ollama_ready():
    ollama_log = open("/tmp/ollama-kaggle.log", "w")
    server = subprocess.Popen([OLLAMA, "serve"], stdout=ollama_log, stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_ready():
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama ne répond pas ; consulter /tmp/ollama-kaggle.log")

subprocess.run([OLLAMA, "pull", "qwen3:8b"], check=True)
subprocess.run([OLLAMA, "ps"], check=True)
print("Modèle prêt")

In [ ]:
os.environ.update({
    "OLLAMA_BASE_URL": "http://127.0.0.1:11434",
    "OLLAMA_MODEL": "qwen3:8b",
    "OLLAMA_TIMEOUT_SECONDS": "180",
    "OLLAMA_NUM_PREDICT": "4096",
    "OLLAMA_NUM_CTX": "8192",
    "OLLAMA_THINK": "false",
})
RESULTS = REPO / "evaluation" / "results"
RESULTS.mkdir(exist_ok=True)
def run_benchmark(n, filename, resume=False):
    cmd = [sys.executable, "-u", "evaluation/run_qwen_evaluation.py",
           "--confirm-live-ollama", "--max-cases", str(n),
           "--output", str(RESULTS / filename)]
    if resume: cmd.append("--resume")
    subprocess.run(cmd, cwd=str(REPO), check=True)
    report = json.loads((RESULTS / filename).read_text(encoding="utf-8"))
    for item in report["results"]:
        score = item.get("semantic_fidelity")
        print(item["id"], "score:", score, "error:", item.get("error"))
    print("Rapport :", RESULTS / filename)
run_benchmark(2, "qwen-smoke-2.json")

## Benchmark complet (facultatif)

Après vérification des deux premiers résultats, passer `RUN_ALL_12` à `True`. Si l'exécution est interrompue et le fichier de rapport existe, passer `RESUME` à `True` ; les cas enregistrés ne seront pas relancés.

In [ ]:
RUN_ALL_12 = False
RESUME = False
if RUN_ALL_12:
    run_benchmark(12, "qwen-full-12.json", resume=RESUME)
else:
    print("Exécution complète désactivée : RUN_ALL_12 = True pour lancer les 12 cas.")